In [1]:
!unzip -q images.zip -d /content/images
!unzip -q depth.zip -d /content/depth

In [3]:
import os

image_dir = "/content/images/images"
depth_dir = "/content/depth/depth"

print("Total imágenes:", len(os.listdir(image_dir)))
print("Total mapas de profundidad:", len(os.listdir(depth_dir)))

Total imágenes: 98
Total mapas de profundidad: 98


In [6]:
!pip install openexr Imath

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.2 MB/s eta 0:00:00


In [7]:
import os
import glob
import shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import numpy as np
import OpenEXR
import Imath
import matplotlib.pyplot as plt
from tqdm import tqdm

Reading function

In [16]:
def read_exr_depth(exr_path):
    exr_file = OpenEXR.InputFile(exr_path)
    dw = exr_file.header()['dataWindow']
    width, height = dw.max.x - dw.min.x + 1, dw.max.y - dw.min.y + 1
    pt = Imath.PixelType(Imath.PixelType.FLOAT)
    depth_str = exr_file.channel('R', pt)
    depth = np.frombuffer(depth_str, dtype=np.float32).reshape((height, width))
    return depth

class DepthDataset(Dataset):
    def __init__(self, image_dir, depth_dir, transform=None):
        self.image_paths = sorted([
            os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(".png")
        ])
        self.depth_paths = sorted([
            os.path.join(depth_dir, f) for f in os.listdir(depth_dir) if f.endswith(".exr")
        ])

        assert len(self.image_paths) == len(self.depth_paths), (
            f"Mismatch: {len(self.image_paths)} images, {len(self.depth_paths)} depth maps"
        )

        self.transform = transform or transforms.Compose([
            transforms.Resize((256, 512)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image = self.transform(image)

        depth = read_exr_depth(self.depth_paths[idx])
        depth = Image.fromarray(depth)
        depth = transforms.Resize((256, 512))(depth)
        depth = transforms.ToTensor()(depth)

        return image, depth


Load Data

In [17]:
from torch.utils.data import DataLoader, random_split

image_dir = "/content/images/images"
depth_dir = "/content/depth/depth"

dataset = DepthDataset(image_dir, depth_dir)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)


U-NET MODEL

In [18]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.upconv4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.upconv3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.upconv2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.upconv1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)


In [21]:
import torch.nn as nn
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet().to(device)

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=5):
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for images, depths in train_loader:
            images, depths = images.to(device), depths.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, depths)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, depths in val_loader:
                images, depths = images.to(device), depths.to(device)
                outputs = model(images)
                loss = criterion(outputs, depths)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

# ✅ Finally, run the training
train_model(model, train_loader, val_loader, criterion, optimizer, device)


Epoch 1: Train Loss = 7.6918, Val Loss = 7.7266
Epoch 2: Train Loss = 6.9566, Val Loss = 6.7517
Epoch 3: Train Loss = 5.6921, Val Loss = 4.6002
Epoch 4: Train Loss = 4.6144, Val Loss = 4.1185
Epoch 5: Train Loss = 3.9645, Val Loss = 3.7507


In [22]:
def compute_metrics(gt, pred):
    """
    Compute standard depth estimation metrics.
    Inputs should be numpy arrays of shape (H, W)
    """
    # To avoid division by zero
    mask = gt > 0
    gt = gt[mask]
    pred = pred[mask]

    # Clamp predictions to avoid log(0) or divide by zero
    pred = np.clip(pred, 1e-6, None)

    abs_rel = np.mean(np.abs(gt - pred) / gt)
    sq_rel = np.mean(((gt - pred) ** 2) / gt)
    rmse = np.sqrt(np.mean((gt - pred) ** 2))
    mae = np.mean(np.abs(gt - pred))

    # Threshold accuracy
    thresh = np.maximum(gt / pred, pred / gt)
    delta1 = (thresh < 1.25).mean()
    delta2 = (thresh < 1.25 ** 2).mean()
    delta3 = (thresh < 1.25 ** 3).mean()

    return {
        "AbsRel": abs_rel,
        "SqRel": sq_rel,
        "RMSE": rmse,
        "MAE": mae,
        "δ1": delta1,
        "δ2": delta2,
        "δ3": delta3
    }

def evaluate_model(model, dataloader, device):
    model.eval()
    metrics_list = []

    with torch.no_grad():
        for images, depths in dataloader:
            images = images.to(device)
            depths = depths.to(device)

            outputs = model(images)
            outputs = outputs.squeeze(1).cpu().numpy()
            depths = depths.squeeze(1).cpu().numpy()

            for i in range(images.size(0)):
                pred = outputs[i]
                gt = depths[i]
                metrics = compute_metrics(gt, pred)
                metrics_list.append(metrics)

    # Compute average over all samples
    avg_metrics = {k: np.mean([m[k] for m in metrics_list]) for k in metrics_list[0]}
    print("\n📊 Evaluation Results:")
    for key, value in avg_metrics.items():
        print(f"{key}: {value:.4f}")

    return avg_metrics

In [23]:
evaluate_model(model, val_loader, device)



📊 Evaluation Results:
AbsRel: 0.8874
SqRel: 11.3735
RMSE: 9.8837
MAE: 5.0030
δ1: 0.4771
δ2: 0.7148
δ3: 0.8390


{'AbsRel': np.float32(0.88736457),
 'SqRel': np.float32(11.37351),
 'RMSE': np.float32(9.883673),
 'MAE': np.float32(5.002976),
 'δ1': np.float64(0.4771043732470018),
 'δ2': np.float64(0.714808557817128),
 'δ3': np.float64(0.838973787548628)}